In [1]:
# imports

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window, hilbert
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import importlib
import utils as ut
importlib.reload(ut)

<module 'utils' from '/Users/itamarmerfeld/Projects/UATR/dev/utils.py'>

In [2]:
# simulation parameters
fs = 128000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
snr_values = [6, 4, 2, 0, -2, -4, -6, -8, -10, -12]  # from 0 dB to -8 dB

# detector args
nperseg = 8000
nfft = None # 2*nperseg
overlap = 0.25
window = 'hanning'
dc = 20
crop_freq = 2000
norm_size = 9

raw_signal = ut.simulate_raw_signal(f0, fs, duration)
F, T, raw_Sxx, raw_phasogram = ut.calc_spectrogram(raw_signal, fs, nfft=nfft, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

s2g_laplacian_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='laplacian', quantization_levels=5)
s2g_wasserstein_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode='wasserstein', quantization_levels=3)
correlation_detector = ut.ClassicDetector(mode="correlation", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
entropy_detector = ut.ClassicDetector(mode="entropy", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
demon_detector = ut.ClassicDetector(mode="demon", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
welch_detector = ut.ClassicDetector(mode="welch", fs=fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)

fig = make_subplots(rows=6,
                    cols=len(snr_values),
                    shared_xaxes=True,
                    vertical_spacing=0.04,
                    column_titles=[f"snr={snr} dB" for snr in snr_values],
                    row_titles=["Welch", "DEMON", "Correlation", "Entropy", "Laplacian", "Wasserstein"])

for i, snr in enumerate(snr_values, start=1):
    rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')

    _F, _T, rx_Sxx, _phasogram = ut.calc_spectrogram(rx, fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

    welch, F_welch = welch_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=welch, x=F_welch, showlegend=False, line=dict(color='blue')), row=1, col=i)

    demon, F_demon = demon_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=demon, x=F_demon, showlegend=False, line=dict(color='red')), row=2, col=i)

    correlation, F_correlation = correlation_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=correlation, x=F_correlation, showlegend=False, line=dict(color='green')), row=3, col=i)

    entropy, F_entropy = entropy_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=entropy, x=F_entropy, showlegend=False, line=dict(color='purple')), row=4, col=i)

    laplacian, F_laplacian = s2g_laplacian_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=laplacian, x=F_laplacian, showlegend=False, line=dict(color='orange')), row=5, col=i)

    wasserstein, F_wasserstein = s2g_wasserstein_detector.get_feature_vector(rx)
    fig.add_trace(go.Scatter(y=wasserstein, x=F_wasserstein, showlegend=False, line=dict(color='cyan')), row=6, col=i)

fig.update_xaxes(title_text='Frequency (Hz)', title_font=dict(size=20), row=6)
fig.update_yaxes(showticklabels=False)

fig.update_layout(
    height=900, width=1600,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)

# Match column/row title offsets and bold them.
# Column titles sit above row 1 at y = 1 + offset; row titles sit to the
# right of the last column at x = 1 + offset (in paper coords).
title_offset = 0.06
column_title_texts = [f"snr={snr} dB" for snr in snr_values]
row_title_texts = ["Welch", "DEMON", "Correlation", "Entropy", "Laplacian", "Wasserstein"]
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
    if ann.text in column_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.y = 1 + title_offset
        ann.yanchor = 'bottom'
    elif ann.text in row_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.x = 1 + title_offset
        ann.xanchor = 'left'
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/simulation_example.svg")  #, scale=10)
fig.show()